<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.5-rag-with-langchain/notebooks/exercises-6.5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.5 — RAG with LangChain
Netsetos GenAI Course v2.0 | Module 6

LCEL, LangGraph, agents, production deployment, LangSmith, advanced retrievers


In [ ]:
# pip install langchain langchain-openai langchain-chroma langchain-text-splitters langgraph rank-bm25 -q
print('Ready for LangChain RAG')


## Ex 1: LCEL RAG Chain


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Index documents
docs = ['RAG grounds LLM responses in external documents.',
        'LangChain uses LCEL for composable chains.',
        'LangGraph adds state machines for self-corrective RAG.']
from langchain_core.documents import Document
documents = [Document(page_content=d) for d in docs]

db = Chroma.from_documents(documents, OpenAIEmbeddings(model='text-embedding-3-small'))
retriever = db.as_retriever(search_kwargs={'k': 2})

# 2. Build LCEL RAG chain
prompt = ChatPromptTemplate.from_template(
    'Answer using ONLY this context:\n{context}\n\nQuestion: {question}')

def format_docs(docs):
    return '\n'.join(d.page_content for d in docs)

rag_chain = (
    {'context': retriever | RunnableLambda(format_docs),
     'question': RunnablePassthrough()}
    | prompt
    | ChatOpenAI(model='gpt-4o-mini')
    | StrOutputParser()
)

# 3. Invoke
result = rag_chain.invoke('What is LCEL?')
print(result)

# 4. Stream
for chunk in rag_chain.stream('What is LangGraph?'):
    print(chunk, end='', flush=True)


## Ex 2: EnsembleRetriever (Hybrid Search)


In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# BM25 + Dense with RRF fusion
bm25 = BM25Retriever.from_documents(documents, k=2)
vector = db.as_retriever(search_kwargs={'k': 2})

ensemble = EnsembleRetriever(
    retrievers=[bm25, vector],
    weights=[0.3, 0.7]  # 30% BM25, 70% dense
)

results = ensemble.invoke('What is LCEL?')
for r in results:
    print(f'  {r.page_content[:60]}...')


## Ex 3: Conversational RAG with History


In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conv_chain = RunnableWithMessageHistory(
    rag_chain, get_history,
    input_messages_key='question',
    history_messages_key='history'
)

# Multi-turn conversation
result1 = conv_chain.invoke(
    {'question': 'What is RAG?'},
    config={'configurable': {'session_id': 'user-1'}}
)
print(f'Turn 1: {result1[:80]}...')

result2 = conv_chain.invoke(
    {'question': 'How does it compare to fine-tuning?'},
    config={'configurable': {'session_id': 'user-1'}}
)
print(f'Turn 2: {result2[:80]}...')


## Ex 4: LangGraph Self-Corrective RAG


In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List

class RAGState(TypedDict):
    question: str
    documents: list
    generation: str
    retries: int

def retrieve(state):
    docs = retriever.invoke(state['question'])
    return {'documents': docs, 'retries': state.get('retries', 0)}

def grade_docs(state):
    # Simple relevance check (production: use LLM grading)
    relevant = [d for d in state['documents'] if len(d.page_content) > 10]
    return {'documents': relevant}

def generate(state):
    context = '\n'.join(d.page_content for d in state['documents'])
    response = ChatOpenAI(model='gpt-4o-mini').invoke(
        f'Answer from context: {context}\nQ: {state["question"]}')
    return {'generation': response.content}

def decide(state):
    if len(state['documents']) > 0:
        return 'generate'
    if state.get('retries', 0) >= 3:
        return 'generate'  # Best effort after 3 retries
    return 'rewrite'

def rewrite(state):
    return {'question': f'More specific: {state["question"]}',
            'retries': state.get('retries', 0) + 1}

workflow = StateGraph(RAGState)
workflow.add_node('retrieve', retrieve)
workflow.add_node('grade', grade_docs)
workflow.add_node('generate', generate)
workflow.add_node('rewrite', rewrite)
workflow.add_edge(START, 'retrieve')
workflow.add_edge('retrieve', 'grade')
workflow.add_conditional_edges('grade', decide,
    {'generate': 'generate', 'rewrite': 'rewrite'})
workflow.add_edge('rewrite', 'retrieve')
workflow.add_edge('generate', END)

graph = workflow.compile()
result = graph.invoke({'question': 'What is LCEL?', 'retries': 0})
print(result['generation'])


## Ex 5: Multi-Tool Agent


In [ ]:
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def search_docs(query: str) -> str:
    '''Search internal documentation.'''
    docs = retriever.invoke(query)
    return '\n'.join(d.page_content for d in docs)

@tool
def calculate(expression: str) -> str:
    '''Calculate a math expression.'''
    try: return str(eval(expression))
    except: return 'Error'

# agent = create_react_agent(
#     model='openai:gpt-4o',
#     tools=[search_docs, calculate]
# )
# result = agent.invoke({'messages': [{'role': 'user', 'content': 'What is LCEL?'}]})

print('Agent pattern:')
print('  1. LLM decides which tool to call')
print('  2. Tool executes, returns result')
print('  3. LLM decides: call another tool or generate answer')
print('  4. Loop until done')


## Ex 6: LangSmith Tracing + Callbacks


In [ ]:
import os
# os.environ['LANGSMITH_TRACING'] = 'true'
# os.environ['LANGSMITH_API_KEY'] = '<your-key>'
# os.environ['LANGSMITH_PROJECT'] = 'rag-lesson-6.5'

from langchain_core.callbacks.base import BaseCallbackHandler

class CostTracker(BaseCallbackHandler):
    def __init__(self):
        self.total_tokens = 0
        self.total_cost = 0.0
    
    def on_llm_end(self, response, **kwargs):
        usage = response.llm_output.get('token_usage', {}) if response.llm_output else {}
        tokens = usage.get('total_tokens', 0)
        self.total_tokens += tokens
        self.total_cost += tokens * 0.00003  # Approximate
        print(f'  [Cost] {tokens} tokens, ${self.total_cost:.4f} cumulative')

tracker = CostTracker()
# result = rag_chain.invoke('What is RAG?', config={'callbacks': [tracker]})
print('LangSmith: set 2 env vars → full tracing, zero code changes')
print('Custom callbacks: on_retriever_end, on_llm_end, on_chain_error')


## Ex 7: Advanced Retrievers


In [ ]:
# ParentDocumentRetriever
# from langchain.retrievers import ParentDocumentRetriever
# from langchain.storage import InMemoryStore
# parent_retriever = ParentDocumentRetriever(
#     vectorstore=db, docstore=InMemoryStore(),
#     child_splitter=RecursiveCharacterTextSplitter(chunk_size=200),
#     parent_splitter=RecursiveCharacterTextSplitter(chunk_size=1000)
# )

# SelfQueryRetriever
# from langchain.retrievers.self_query.base import SelfQueryRetriever
# retriever = SelfQueryRetriever.from_llm(llm, vectorstore, document_content_description, metadata_field_info)

print('Advanced Retrievers:')
for name, desc in [
    ('EnsembleRetriever', 'BM25 + dense with RRF (hybrid search)'),
    ('MultiQueryRetriever', 'LLM generates 3-5 query variations'),
    ('ContextualCompression', 'Compresses docs to relevant portions'),
    ('ParentDocument', 'Index small chunks, return full parents'),
    ('SelfQuery', 'NL → metadata filters + semantic query')]:
    print(f'  {name:25s}: {desc}')


## Ex 8: Incremental Indexing


In [ ]:
# from langchain.indexes import SQLRecordManager
# from langchain_core.indexing.api import index

# record_manager = SQLRecordManager('my_docs', db_url='sqlite:///records.db')
# record_manager.create_schema()

# result = index(
#     docs_source=documents,
#     record_manager=record_manager,
#     vector_store=db,
#     cleanup='incremental',
#     source_id_key='source'
# )
# print(result)  # {num_added: 3, num_updated: 0, num_skipped: 0, num_deleted: 0}

print('RecordManager cleanup modes:')
for mode, desc in [
    ('None', 'Dedup only, no deletions'),
    ('incremental', 'Delete stale for current batch source IDs'),
    ('full', 'Delete everything not in current batch')]:
    print(f'  {mode:15s}: {desc}')
print('\nWithout this: re-indexing creates duplicates!')


---
## Recap
LangChain v0.3+ architecture: langchain-core + langchain-{provider} packages. LCEL: pipe operator creates RunnableSequence with automatic streaming/async/batch. LangGraph: StateGraph for self-corrective RAG (retrieve → grade → generate or rewrite → retry). Agents: @tool + create_react_agent() replaces deprecated AgentExecutor. Production: .astream_events() for streaming, RedisCache for caching, .with_fallbacks() for resilience. LangSmith: 2 env vars for full tracing + custom callbacks + evaluation. Advanced retrievers: EnsembleRetriever (hybrid, production default), ParentDocument (chunk-size tradeoff), SelfQuery (NL → metadata filters), RecordManager (incremental indexing). LlamaIndex for data, LangChain for orchestration — use both together.
